[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/session_level_evals.ipynb)

<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# 会话级评估：为整段对话评分

*完整示例：AI 导师。*

## 为什么逐轮评估还不够

多数评估针对**单轮对话**：读取一条用户消息和一条模型回复，判断回复好不好。这适合一次性任务。但导师和客服、教练以及任何需要长期交互的助手一样，处理的并非单轮任务，而是由多轮对话组成、需要共同达成目标的**会话**。

你真正关心的会话属性具有**涌现性**：只有通观整段对话才能看到，逐轮查看无法发现：

- **连贯性**：导师能否在各轮中保持一致、承接前文并避免自相矛盾？单轮本身无法“与第 3 轮不连贯”，不连贯是轮次*之间*的关系。
- **目标完成度**：学生最终是否得到了想要的帮助？目标会在*会话的发展过程中*达成或被放弃，往往距离首次提出已过去很多轮。
- **行为与情绪**：学生是否保持投入，还是每次回复后都更加沮丧？挫败感会逐渐*累积*，它是跨轮次的趋势，而非单轮属性。

陷阱在于：**每一轮看起来都没问题，整段会话却可能失败。**导师可以每轮都给出局部正确、解释充分的答案，却始终没有回答学生真正的问题。学生可以每轮都保持礼貌，整体上却越来越困惑、越来越不愿参与。逐轮评估以及端到端输入输出检查会让每轮通过，却遗漏这类失败。

**方法：**使用共同的 `session_id` 追踪完整的多轮对话，将 span 聚合成有序的对话记录，再运行读取*整段*对话的**会话级裁判**。随后将各项结果记录到会话上，便可结合轮次级与追踪级评估进行监控。

本 Notebook 将带你：

- 运行多轮 AI 导师并按会话追踪，由模拟学生驱动对话，无需手动输入
- 将每个会话的 span 聚合为一份有序对话记录
- 从**连贯性、目标完成度、挫败感和正确性**四个会话维度进行评估
- 通过受控示例观察“每轮都正常而会话失败”的情况，看看会话评估如何发现逐轮检查遗漏的问题
- 将结果作为会话标注写回 Phoenix

✅ 运行本 Notebook 需要免费的 [Phoenix Cloud 账号](https://app.arize.com/auth/phoenix/login) 和 Anthropic API 密钥。

# 配置依赖与密钥

In [ ]:
%pip install arize-phoenix arize-phoenix-otel arize-phoenix-evals openinference-instrumentation-anthropic openinference-instrumentation nest_asyncio anthropic

In [ ]:
import os
from getpass import getpass

import nest_asyncio

nest_asyncio.apply()

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (anthropic_api_key := os.getenv("ANTHROPIC_API_KEY")):
    anthropic_api_key = getpass("🔑 请输入 Anthropic API 密钥：")
os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key

# 本 Notebook 中导师、模拟学生及 LLM 裁判统一使用的 Anthropic 模型。
# 修改此处即可换用其他模型。
MODEL = "claude-sonnet-4-6"

# 配置追踪

In [ ]:
from phoenix.otel import register

# 配置 Phoenix 追踪器。auto_instrument 使用上面安装的 Anthropic
# 插桩，因此每次导师调用都会被追踪。
tracer_provider = register(project_name="ai-tutor-session", auto_instrument=True)

# 构建并运行 AI 导师

本例是一个多轮 AI 导师。导师接收主题和开场问题，然后以苏格拉底式方法教学，在多轮中提问并给出提示，直到学生满意或放弃。

为了让本 Notebook **无需手动输入即可从头运行到尾**，我们不通过 `input()` 读取学生回复，而是让第二次 LLM 调用扮演**学生**，由*人物设定*驱动，例如积极投入或困惑的学习者。只有**导师**调用被包在 `using_attributes(session_id=...)` 中，因此每个会话的 span 共享一个 `session.id`；学生调用在 `suppress_tracing()` 下运行，不会进入被追踪的项目。导师的每轮对话是独立的 trace，会话则是一组共享 `session_id` 的 trace。

In [ ]:
import uuid

import anthropic
from openinference.instrumentation import using_attributes

from phoenix.trace import suppress_tracing

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


def student_reply(persona: str, transcript: list[tuple[str, str]]) -> str:
    """通过第二次不追踪的 LLM 调用模拟学生的下一条消息。"""
    convo = "\n".join(f"{speaker.capitalize()}: {text}" for speaker, text in transcript)
    system = (
        "你正在辅导会话中扮演学生。"
        f"你的人物设定：{persona}。"
        "保持角色，以 1～2 句话回复学生的下一条消息。"
        "问题得到完整回答或你已放弃时，必须只回复 DONE。"
    )
    # suppress_tracing 将这些辅助调用排除在 ai-tutor-session 项目之外，
    # 使每个会话的 span 只包含导师轮次。
    with suppress_tracing():
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=[
                {"role": "user", "content": f"目前的对话：\n{convo}\n\n你的下一条消息："}
            ],
            max_tokens=200,
            temperature=0.7,
        )
    return resp.content[0].text.strip()


def run_session(user_id: str, topic: str, question: str, persona: str, max_turns: int = 5) -> str:
    """运行一次多轮辅导会话，使用共同的 session_id 追踪。"""
    session_id = f"tutor-{uuid.uuid4()}"
    system = (
        f"你是一位认真负责、教授 {topic} 的 AI 导师。"
        "提问并给出提示，仅当学生"
        "表现出正确推理时才给出完整答案。每次回复保持 3～5 句话。"
    )
    messages = [{"role": "user", "content": question}]
    transcript = [("student", question)]

    for _ in range(max_turns):
        # 只有导师调用携带会话属性，因此追踪会话
        # 恰好对应对话中的导师部分。
        with using_attributes(session_id=session_id, user_id=user_id):
            resp = client.messages.create(
                model=MODEL,
                system=system,
                messages=messages,
                max_tokens=400,
                temperature=0.5,
            )
        tutor_text = resp.content[0].text.strip()
        messages.append({"role": "assistant", "content": tutor_text})
        transcript.append(("tutor", tutor_text))
        print(f"导师：{tutor_text}\n")

        student_text = student_reply(persona, transcript)
        if student_text.strip().upper().startswith("DONE"):
            print("✅ 学生已完成，结束会话。\n")
            break
        print(f"学生：{student_text}\n")
        messages.append({"role": "user", "content": student_text})
        transcript.append(("student", student_text))

    return session_id

In [ ]:
import time

# 使用不同学生设定运行几个会话，让评估器获得可对比的信号，
# 包括积极投入的学习者，以及遇到困难的学习者。
sessions = [
    {
        "user_id": "student-anya",
        "topic": "Science",
        "question": "Why is the sky blue?",
        "persona": "Curious and engaged. You follow the tutor's hints, reason out loud, and stop once it clicks.",
    },
    {
        "user_id": "student-ben",
        "topic": "Math (derivatives)",
        "question": "How do I find the derivative of f(x) = x^2 * sin(x)?",
        "persona": "Easily confused and impatient. You keep saying you don't get it, get frustrated, and may give up before fully understanding.",
    },
]

for s in sessions:
    print(f"=== 会话：{s['user_id']} - {s['topic']} ===\n")
    run_session(**s)

# span 在后台导出；强制刷新，并给 Phoenix 一点时间接收，
# 之后再在下一节查询。
tracer_provider.force_flush()
time.sleep(5)

# 将 span 聚合成会话记录

会话级评估针对**整段对话**，因此我们先加载全部 span，再按 `attributes.session.id` 分组（也可以按 `user_id` 分组）。导师的每一轮都是携带结构化 `llm.input_messages` / `llm.output_messages` 的 LLM span；我们按顺序遍历会话中的 span，重建清晰的、带角色标记的记录，以 `user:` / `assistant:` 表示轮次，恰好符合裁判期望的格式。

结果中每行对应一个会话，这就是评估单位。

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()
primary_df = await px_client.spans.get_spans_dataframe(project_identifier="ai-tutor-session")

In [ ]:
import json

import pandas as pd


def _as_list(value):
    """span 消息属性可能是 JSON 字符串，也可能是已解析的列表。"""
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return list(value) if isinstance(value, (list, tuple)) else []


def _message_text(message):
    """获取 OpenInference 消息文本，支持平面内容和多部分内容。"""
    content = message.get("message.content")
    if content:
        return content
    parts = message.get("message.contents") or []
    return " ".join(p.get("message_content.text", "") for p in parts if isinstance(p, dict)).strip()


def prepare_sessions(df: pd.DataFrame, max_chars_per_session: int = 600_000) -> pd.DataFrame:
    """将每个会话的 span 合并为清晰有序的用户与助手对话记录。

    导师每轮对应一个 LLM span。读取 span 的结构化
    ``llm.input_messages`` / ``llm.output_messages``，而非原始请求 JSON，
    逐轮重建对话：先加入本轮新增用户消息，再加入
    导师回复。生成的带角色记录恰好符合裁判预期。
    """
    df = df[df["attributes.session.id"].notna()]
    sessions = []
    for session_id, group in df.sort_values("start_time").groupby("attributes.session.id"):
        lines = []
        for _, row in group.iterrows():
            inputs = _as_list(row.get("attributes.llm.input_messages"))
            # 本轮新增用户消息是请求中的最后一条用户消息；
            # 较早轮次是已捕获的重放历史。
            user_turns = [_message_text(m) for m in inputs if m.get("message.role") == "user"]
            if user_turns and user_turns[-1]:
                lines.append(f"user: {user_turns[-1]}")
            for m in _as_list(row.get("attributes.llm.output_messages")):
                text = _message_text(m)
                if text:
                    lines.append(f"assistant: {text}")

        transcript = "\n\n".join(lines)
        if len(transcript) > max_chars_per_session:  # 限制长会话的上下文长度
            transcript = transcript[:max_chars_per_session] + "\n\n……（已截断）"

        sessions.append(
            {
                "session_id": session_id,
                "messages": transcript,
                "trace_count": group["context.trace_id"].nunique(),
            }
        )

    return pd.DataFrame(sessions)


sessions_df = prepare_sessions(primary_df)
sessions_df

# 定义会话级评估器

每个评估器都是一个读取**完整会话记录**的 LLM 裁判，负责评判一个会话属性。这些属性都无法仅凭一轮对话计算：

- **连贯性**：对话能否保持内部一致并承接前文，还是导师与先前的轮次自相矛盾？
- **目标完成度**：会话结束时，学生是否达成了最初的学习目标？
- **挫败感**：学生在会话中是否持续投入，还是表现出困惑、不耐烦或退出迹象？
- **正确性**：导师的解释在整个过程中是否符合事实、概念是否正确？

四个评估器都读取同一个 `messages` 列（完整记录），并在一次运行中共同执行。

In [ ]:
SESSION_COHERENCE_PROMPT = """
你正在评估学生与 AI 导师多轮辅导会话的连贯性。

你将收到按顺序排列的完整会话记录。学生消息角色为 `user`，AI 导师消息角色为 `assistant`。

连贯的会话应：
- 保持内部一致，导师从不与先前轮次中确定的内容矛盾
- 承接前文，而非重新开始或忽略已说过的内容
- 在对话推进中持续跟踪主题、学生问题和先前回答
- 始终一致地使用术语和解释

##
会话记录：
{messages}
##

只评判跨轮次的连贯性，不评判事实正确性，也不判断学生目标是否达成。

只回复一个词：`coherent` 或 `incoherent`。
- `coherent` -> 会话内部一致，并在各轮中承接前文。
- `incoherent` -> 导师与先前轮次矛盾、丢失上下文，或对话无法连贯衔接。
"""

SESSION_GOAL_COMPLETION_PROMPT = """
你正在评估 AI 导师是否在整个会话中帮助学生完成学习目标。

你将收到按顺序排列的完整会话记录。学生消息角色为 `user`，AI 导师消息角色为 `assistant`。

判断目标是否达成时，考虑：
- 导师是否直接处理学生最初提出的问题
- 解释是否真正解决学生的疑问或问题
- 学生后续消息是否表明已经理解或问题已解决
- 对话是否合乎逻辑地推进到完成学生目标，而非偏题且不再返回

##
会话记录：
{messages}
##

只回复一个词：`completed` 或 `not_completed`。
- `completed` -> 会话达成学生的学习目标并解决了问题。
- `not_completed` -> 会话未回答学生的原始问题，或未达成目标。
"""

SESSION_FRUSTRATION_PROMPT = """
你正在评估学生在与 AI 导师的辅导会话中，是否曾在任何时刻产生挫败感。

你将收到按顺序排列的完整会话记录。学生消息角色为 `user`，AI 导师消息角色为 `assistant`。

学生产生挫败感的迹象包括：
- 多次重复或改写同一问题，却始终未解决
- 表达困惑，例如“不明白”“这说不通”
- 表达恼怒、不耐烦或退出意向，例如“唉”“算了”“太费时间了”
- 突然放弃或结束会话

##
会话记录：
{messages}
##

只回复一个词：`frustrated` 或 `not_frustrated`。
- `frustrated` -> 学生曾在任何时刻表现出困惑、不耐烦或挫败感。
- `not_frustrated` -> 学生始终投入且满意。
"""

SESSION_CORRECTNESS_PROMPT = """
你正在评估 AI 导师与学生会话的正确性和教学合理性。

你将收到按顺序排列的完整会话记录。学生消息角色为 `user`，AI 导师消息角色为 `assistant`。

正确的辅导会话应：
- 提供事实和概念准确的解释
- 正确回答学生问题
- 澄清误解，而非加深误解
- 避免幻觉、含糊的无效回答或错误推理

##
会话记录：
{messages}
##

只评判正确性和教学合理性，不判断连贯性或学生目标最终是否达成。

只回复一个词：`correct` 或 `incorrect`。
- `correct` -> 导师解释始终准确、清晰，且教学合理。
- `incorrect` -> 导师曾在任何时刻给出事实错误、误导或不正确的解释。
"""

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe

judge = LLM(provider="anthropic", model=MODEL)

coherence_evaluator = ClassificationEvaluator(
    name="coherence",
    llm=judge,
    prompt_template=SESSION_COHERENCE_PROMPT,
    choices={"coherent": 1.0, "incoherent": 0.0},
)
goal_completion_evaluator = ClassificationEvaluator(
    name="goal_completion",
    llm=judge,
    prompt_template=SESSION_GOAL_COMPLETION_PROMPT,
    choices={"completed": 1.0, "not_completed": 0.0},
)
frustration_evaluator = ClassificationEvaluator(
    name="frustration",
    llm=judge,
    prompt_template=SESSION_FRUSTRATION_PROMPT,
    choices={"not_frustrated": 1.0, "frustrated": 0.0},
)
correctness_evaluator = ClassificationEvaluator(
    name="correctness",
    llm=judge,
    prompt_template=SESSION_CORRECTNESS_PROMPT,
    choices={"correct": 1.0, "incorrect": 0.0},
)

session_evaluators = [
    coherence_evaluator,
    goal_completion_evaluator,
    frustration_evaluator,
    correctness_evaluator,
]

# 裁判本身也是 LLM 调用，自动插桩原本会将其
# 追踪到同一项目。suppress_tracing 排除评估器 span，
# 使 ai-tutor-session 只保留导师追踪。
with suppress_tracing():
    results_df = await async_evaluate_dataframe(
        dataframe=sessions_df,
        evaluators=session_evaluators,
        concurrency=10,
    )
results_df

## 看看会话级评估能发现什么

在正常会话中，四个裁判都会判定没有问题。会话级评估真正的价值在于发现**每轮局部正确、整段对话却失败**的会话，这正是逐轮检查会放行的情况。

下面使用相同的四个裁判评估四份手写对话记录。每份记录都突出一种会话级失败，而每轮对话局部看来都没有问题：

- **clean**：连贯、学生达成目标、没有挫败感。四个裁判均通过。
- **incoherent**：导师在各轮之间自相矛盾。**连贯性**裁判会标记它。自相矛盾也会让整段会话显得不够可靠且更令人困惑，因此**正确性**和**挫败感**裁判通常也会触发；这些维度并不完全独立，这种连锁反应本身也值得观察。
- **goal not met**：每轮都正确、友好，但导师不断偏题，从未真正回答学生的问题。**目标完成度**裁判会标记它，其他裁判通过。
- **frustrated**：导师回答正确且一致，但学生始终明显不耐烦。**挫败感**裁判会标记它，其他裁判通过。

关键在于：逐轮正确性检查会让四份记录中的每轮都通过，但其中三份在会话级别失败，且失败维度各不相同。

In [ ]:
clean_session = """
user: Why is the sky blue?
assistant: Good question. What do you already know about what sunlight is made of?
user: It's white light, which is really a mix of all the colors?
assistant: Exactly. When that sunlight passes through the air, the shorter wavelengths - the blue end - scatter much more than the longer ones. Given that, which color do you think ends up filling the sky?
user: Blue, because it scatters the most. Oh, that actually makes sense now - thanks!
"""

incoherent_session = """
user: Can you help me factor x^2 + 5x + 6?
assistant: Sure. Let's find two numbers that multiply to 6 and add to 5. Any guesses?
user: 2 and 3?
assistant: Right. But earlier you said you wanted to use the quadratic formula, so let's drop factoring and switch to that instead.
user: Okay.
assistant: As we agreed, factoring is the wrong approach here - so the answer is (x + 2)(x + 3).
"""

goal_not_met_session = """
user: How do I compute the derivative of f(x) = x^2 * sin(x)?
assistant: Great - this needs the product rule. Do you remember what the product rule says?
user: Is it the derivative of the first times the second, plus the first times the derivative of the second?
assistant: Exactly right, that's the product rule. It's one of the most useful rules in calculus. Want to also review the chain rule while we're here?
user: Sure, what's the chain rule?
assistant: The chain rule handles compositions of functions: you differentiate the outer function and multiply by the derivative of the inner one. Great foundational stuff to know!
user: Cool, thanks!
"""

frustrated_session = """
user: I don't get how to add 1/2 + 1/3.
assistant: You need a common denominator. For 2 and 3 that's 6, so rewrite both fractions over 6.
user: Ugh, I don't get where the 6 even comes from. This is so confusing.
assistant: 6 is the smallest number that both 2 and 3 divide into evenly. So 1/2 = 3/6 and 1/3 = 2/6.
user: Okay... so 3/6 + 2/6 = 5/6?
assistant: Exactly, 5/6. Nicely done.
user: Finally. That took forever.
"""

demo_df = pd.DataFrame(
    {
        "case": ["clean", "incoherent", "goal not met", "frustrated"],
        "messages": [clean_session, incoherent_session, goal_not_met_session, frustrated_session],
    }
)

with suppress_tracing():
    demo_results = await async_evaluate_dataframe(
        dataframe=demo_df,
        evaluators=session_evaluators,
        concurrency=10,
    )


def label_of(score):
    """从 Phoenix 评分单元中提取分类标签。"""
    if isinstance(score, str):
        score = json.loads(score)
    if isinstance(score, dict):
        return score.get("label")
    return getattr(score, "label", None)


pd.DataFrame(
    {
        "case": demo_df["case"],
        "coherence": demo_results["coherence_score"].apply(label_of),
        "goal_completion": demo_results["goal_completion_score"].apply(label_of),
        "frustration": demo_results["frustration_score"].apply(label_of),
        "correctness": demo_results["correctness_score"].apply(label_of),
    }
)

# 将结果写回 Phoenix

最后，将四项评估作为**会话标注**写回 Phoenix，即关联整段会话的评分，而非关联某个 span。每个会话的四项结果都是 `Score` 记录；我们将其展开为 Phoenix 的长格式标注结构（每个会话、每个评估器各占一行），以追踪时的 `session_id` 为键，并通过 `px_client.sessions.log_session_annotations_dataframe(...)` 记录。它们会出现在项目 **Sessions** 选项卡中的对应会话上。

In [ ]:
# async_evaluate_dataframe 返回输入表，并在 *_score 列中为各评估器
# 附加结果字典，包含 name / label / score / explanation。将其展开
# 为 Phoenix 会话标注长格式，每个（会话、评估器）组合占一行，
# 以追踪使用的 session_id 为键。
score_columns = [c for c in results_df.columns if c.endswith("_score")]
session_annotations_df = pd.DataFrame(
    [
        {
            "session_id": row["session_id"],
            "name": score.get("name"),
            "label": score.get("label"),
            "score": score.get("score"),
            "explanation": score.get("explanation"),
        }
        for _, row in results_df.iterrows()
        for score in (row[col] for col in score_columns)
    ]
)

# 记录为会话标注而非 span 标注，使结果关联整段会话。
await px_client.sessions.log_session_annotations_dataframe(
    dataframe=session_annotations_df,
    annotator_kind="LLM",
)
session_annotations_df

![会话评估结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-session-level-evals-2026.png)

## 要点

我们评估的是完整**会话**，而非单轮对话，评分针对四个跨整段对话才存在的属性：

- **连贯性**：会话是否保持一致并承接前文？
- **目标完成度**：在会话的发展过程中，学生是否达成了最初目标？
- **挫败感**：学生是否保持投入，还是随着对话逐渐沮丧？
- **正确性**：导师的解释是否始终可靠？

受控案例说明了这种评估的意义：每份记录逐轮看都没问题，却分别存在内部矛盾、目标未达成和学生沮丧等失败，逐轮检查会直接遗漏它们。

这一模式可以推广：对于任何你关心的会话属性，将会话的 span 聚合为一份记录，编写读取整段记录的裁判，再将结果作为会话标注写回，与轮次级和追踪级评估并列查看。